In [110]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

e:\chat_bot_lance_DB\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [109]:
import base64
from io import BytesIO
from PIL import Image

In [124]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler("airline_assistant.log"),
        logging.StreamHandler()
    ],
    force=True
)

logger = logging.getLogger(__name__)

### Initialization of OPEN_AI Model

In [111]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-6-luna"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [112]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

### Initialization of SQL Lite DataBase

In [4]:
import sqlite3
import pandas as pd

In [ ]:
### Create  database

conn = sqlite3.connect("airline_database.db")

cursor = conn.cursor()

### Create a flights table

cursor.execute("""
CREATE TABLE IF NOT EXISTS flights (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    departure_city TEXT NOT NULL,
    destination_city TEXT NOT NULL,
    price_euro REAL NOT NULL,
    airline_operator TEXT NOT NULL
)
""")

conn.commit()

In [ ]:
### Insert airline data

flights = [
    ("Berlin", "London", 120.00, "Lufthansa"),
    ("Berlin", "Paris", 95.00, "Air France"),
    ("Frankfurt", "London", 150.00, "British Airways"),
    ("Frankfurt", "Paris", 110.00, "Lufthansa"),
    ("Munich", "Rome", 130.00, "ITA Airways"),
    ("Munich", "Madrid", 145.00, "Lufthansa"),
    ("Hamburg", "Amsterdam", 85.00, "KLM"),
    ("Berlin", "Rome", 160.00, "Ryanair"),
    ("Frankfurt", "Madrid", 175.00, "Iberia"),
    ("Hamburg", "London", 105.00, "Eurowings"),
    ("Munich", "Paris", 115.00, "Air France"),
    ("Berlin", "Amsterdam", 90.00, "KLM"),
]

cursor.executemany("""
INSERT INTO flights
(departure_city, destination_city, price_euro, airline_operator)
VALUES (?, ?, ?, ?)
""", flights)

conn.commit()

In [9]:
df = pd.read_sql_query("SELECT * FROM flights", conn)

display(df)

,id,departure_city,destination_city,price_euro,airline_operator
0,1,Berlin,London,120.0,Lufthansa
1,2,Berlin,Paris,95.0,Air France
2,3,Frankfurt,London,150.0,British Airways
3,4,Frankfurt,Paris,110.0,Lufthansa
4,5,Munich,Rome,130.0,ITA Airways
5,6,Munich,Madrid,145.0,Lufthansa
6,7,Hamburg,Amsterdam,85.0,KLM
7,8,Berlin,Rome,160.0,Ryanair
8,9,Frankfurt,Madrid,175.0,Iberia
9,10,Hamburg,London,105.0,Eurowings


In [ ]:
### test: Find flights from Berlin

cursor.execute("""
SELECT * FROM flights
WHERE departure_city = 'Berlin'
""")

results = cursor.fetchall()

for flight in results:
    print(flight)

(1, 'Berlin', 'London', 120.0, 'Lufthansa')
(2, 'Berlin', 'Paris', 95.0, 'Air France')
(8, 'Berlin', 'Rome', 160.0, 'Ryanair')
(12, 'Berlin', 'Amsterdam', 90.0, 'KLM')


In [101]:
conn.close()

In [144]:
DB = "airline_database.db"

def get_ticket_price(departure_city, destination_city):
    destination_city = destination_city.title()
    departure_city = departure_city.title()

    logger.info(f"Searching flight: {departure_city} -> {destination_city}")

    print(f"DATABASE TOOL CALLED: Getting price for {departure_city} to {destination_city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("""
                        SELECT price_euro, airline_operator
                        FROM flights
                        WHERE destination_city = ?
                        AND departure_city = ?
                       """, 
                      (destination_city, departure_city))

        results = cursor.fetchall()

        if results:

            logger.info(f"Flight found with price: €{results}")

            return f"Flight from {departure_city} to {destination_city} costs €{results[0]}."

        logger.warning(f"No flight found: {departure_city} -> {destination_city}")

        return "No price data available for this route."

In [145]:
get_ticket_price("berlin", "LONDON")

2026-11-23 14:49:49,486 | INFO | Searching flight: Berlin -> London


DATABASE TOOL CALLED: Getting price for Berlin to London


2026-11-23 14:49:49,872 | INFO | Flight found with price: €[(120.0, 'Lufthansa')]


"Flight from Berlin to London costs €(120.0, 'Lufthansa')."

### Voice Model

In [107]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

### Image Generation Model

In [108]:
def artist(city):
    image_response = openai.images.generate(
            model="gpt-image-1-mini",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [116]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "departure_city": {
                "type": "string",
                "description": "The city that the customer travel from",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'},
     'departure_city': {'type': 'string',
      'description': 'The city that the customer travel from'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [142]:
def chat(history):

    logger.info("New chat started")

    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history

    logger.info("Sending messages to OpenAI")

    response = openai.chat.completions.create(model=MODEL, reasoning_effort="none", messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":

        logger.info("OpenAI requested a tool call")

        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)

        logger.info("Sending tool results back to OpenAI")

        response = openai.chat.completions.create(model=MODEL, reasoning_effort="none", messages=messages, tools=tools)

    reply = response.choices[0].message.content

    logger.info("OpenAI generated final response")

    history += [{"role":"assistant", "content":reply}]

    logger.info("Generating voice")

    voice = talker(reply)

    if cities:

        logger.info(f"Generating image for {cities[0]}")
        image = artist(cities[0])
    
    logger.info("Chat finished successfully")
    return history, voice, image

In [ ]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:

        logger.info(f"Tool called: {tool_call.function.name}")

        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)

            print(arguments)

            destination_city = arguments.get('destination_city')
            print(destination_city)
            departure_city = arguments.get('departure_city') 
            print(departure_city)

            logger.info(f"Destination: {destination_city}")
            logger.info(f"Departure: {departure_city}")

            cities.append(destination_city)
            price_details = get_ticket_price(departure_city, destination_city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

In [ ]:
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

2026-11-23 14:46:43,865 | INFO | HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-11-23 14:46:45,670 | INFO | HTTP Request: GET http://127.0.0.1:7863/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-11-23 14:46:45,703 | INFO | HTTP Request: HEAD http://127.0.0.1:7863/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


2026-11-23 14:46:45,899 | INFO | HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
2026-11-23 14:46:45,986 | INFO | HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-11-23 14:47:18,968 | INFO | New chat started
2026-11-23 14:47:19,018 | INFO | Sending messages to OpenAI
2026-11-23 14:47:22,036 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-23 14:47:22,063 | INFO | OpenAI generated final response
2026-11-23 14:47:22,065 | INFO | Generating voice
2026-11-23 14:47:23,163 | INFO | HTTP Request: POST https://api.openai.com/v1/audio/speech "HTTP/1.1 200 OK"
2026-11-23 14:47:23,784 | INFO | Chat finished successfully
2026-11-23 14:47:43,188 | INFO | New chat started
2026-11-23 14:47:43,217 | INFO | Sending messages to OpenAI
2026-11-23 14:47:44,662 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-2